# SVM Model Training (Out-of-Core)

This notebook trains a Support Vector Machine (SVM) on the preprocessed CICEVSE2024 dataset using `SGDClassifier` and chunking to avoid Out-Of-Memory (OOM) errors.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from tqdm.notebook import tqdm

%matplotlib inline

## 1. Setup Data Paths and Classes

In [ ]:
DATA_DIR = '../../../data/processed'

X_train_path = os.path.join(DATA_DIR, 'X_train.csv')
y_train_path = os.path.join(DATA_DIR, 'y_train.csv')
X_val_path = os.path.join(DATA_DIR, 'X_val.csv')
y_val_path = os.path.join(DATA_DIR, 'y_val.csv')

print("Loading targets to determine classes...")
y_train_full = pd.read_csv(y_train_path)

classes_binary = np.array(sorted(y_train_full["Label_Binary"].unique()))
classes_multi = np.array(sorted(y_train_full["Label_Multiclass"].unique()))

total_samples = len(y_train_full)
del y_train_full

print(f"Binary classes: {classes_binary}")
print(f"Multiclass classes: {classes_multi}")
print(f"Total training samples: {total_samples}")

## 2. Train Models (Chunked)
Using SGDClassifier with `loss='hinge'` to approximate a Linear SVM.

In [ ]:
model_binary = SGDClassifier(loss='hinge', random_state=42)
model_multi = SGDClassifier(loss='hinge', random_state=42)

chunk_size = 100000
total_chunks = (total_samples // chunk_size) + (1 if total_samples % chunk_size != 0 else 0)

X_chunker = pd.read_csv(X_train_path, chunksize=chunk_size)
y_chunker = pd.read_csv(y_train_path, chunksize=chunk_size)

with tqdm(total=total_samples, desc="Processing Rows") as pbar:
    for X_chunk, y_chunk in zip(X_chunker, y_chunker):
        model_binary.partial_fit(X_chunk, y_chunk["Label_Binary"], classes=classes_binary)
        model_multi.partial_fit(X_chunk, y_chunk["Label_Multiclass"], classes=classes_multi)
        pbar.update(len(X_chunk))

print("Training Complete!")

## 3. Evaluation Setup

In [ ]:
def evaluate_model(model, X_path, y_path, title_prefix="", is_multiclass=False):
    # Load entirely into memory for evaluation assuming val/test set is small enough
    X = pd.read_csv(X_path)
    y = pd.read_csv(y_path)
    y = y["Label_Multiclass"] if is_multiclass else y["Label_Binary"]
    
    preds = model.predict(X)
    
    print(f"--- {title_prefix} Classification Report ---")
    print(classification_report(y, preds, zero_division=0))
    
    cm = confusion_matrix(y, preds)
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{title_prefix} Confusion Matrix")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    return preds

## 4. Evaluate Models

In [ ]:
_ = evaluate_model(model_binary, X_val_path, y_val_path, title_prefix="Binary Validation")

In [ ]:
_ = evaluate_model(model_multi, X_val_path, y_val_path, title_prefix="Multiclass Validation", is_multiclass=True)

## 5. Save Models

In [ ]:
SAVE_DIR = '../../../saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)
joblib.dump(model_binary, os.path.join(SAVE_DIR, "svm_model_binary.pkl"))
joblib.dump(model_multi, os.path.join(SAVE_DIR, "svm_model_multiclass.pkl"))
print("Models saved successfully!")